In [7]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
np.random.seed(42)


In [ ]:
# loading orders data 
orders = pd.read_csv("../output/orders.csv")

required_cols = {
    "order_id",
    "sku_id",
    "region",
    "payment_type",
    "delivery_status",
    "delivery_days",
    "order_quantity"
}
missing = required_cols - set(orders.columns)
assert not missing, f"orders.csv missing columns: {missing}"

orders.head()


,order_id,order_date,sku_id,region,payment_type,delivery_days,delivery_status,campaign_applied,order_quantity
0,ORD000000104,0,SKU0002,south,COD,4,DELIVERED,False,1
1,ORD000000103,0,SKU0002,south,COD,3,DELIVERED,False,1
2,ORD000000102,0,SKU0002,south,COD,2,DELIVERED,False,1
3,ORD000000101,0,SKU0002,south,COD,1,DELIVERED,False,1
4,ORD000000100,0,SKU0002,north,PREPAID,5,DELIVERED,False,1


In [ ]:
# cod and rto flags
df = orders.copy()

df["is_cod"] = df["payment_type"] == "COD"
df["is_rto"] = df["delivery_status"] == "RTO"

df[["sku_id", "region", "is_cod", "is_rto"]].head()


,sku_id,region,is_cod,is_rto
0,SKU0002,south,True,False
1,SKU0002,south,True,False
2,SKU0002,south,True,False
3,SKU0002,south,True,False
4,SKU0002,north,False,False


In [ ]:
# aggregate COD Metrics 
grouped = df.groupby(["sku_id", "region"])

cod_metrics = grouped.agg(
    total_orders=("order_id", "count"),
    cod_orders=("is_cod", "sum"),
    cod_rto_orders=("is_rto", lambda x: ((df.loc[x.index, "is_cod"]) & x).sum())
).reset_index()

cod_metrics.head()


,sku_id,region,total_orders,cod_orders,cod_rto_orders
0,SKU0001,east,13921,6957,2889
1,SKU0001,north,13296,6485,2498
2,SKU0001,south,12810,6046,2105
3,SKU0001,west,10932,5079,1622
4,SKU0002,east,13136,6567,2710


In [21]:
# cod intelligence
cod_metrics["cod_share"] = cod_metrics["cod_orders"] / cod_metrics["total_orders"]

cod_metrics["cod_rto_rate"] = np.where(
    cod_metrics["cod_orders"] > 0,
    cod_metrics["cod_rto_orders"] / cod_metrics["cod_orders"],
    0.0
)

cod_metrics["cod_success_rate"] = 1 - cod_metrics["cod_rto_rate"]

cod_metrics.head()


,sku_id,region,total_orders,cod_orders,cod_rto_orders,cod_share,cod_rto_rate,cod_success_rate
0,SKU0001,east,13921,6957,2889,0.499749,0.415265,0.584735
1,SKU0001,north,13296,6485,2498,0.487741,0.385197,0.614803
2,SKU0001,south,12810,6046,2105,0.471975,0.348164,0.651836
3,SKU0001,west,10932,5079,1622,0.464599,0.319354,0.680646
4,SKU0002,east,13136,6567,2710,0.499924,0.412669,0.587331


In [22]:
# cod risk bueckts 
def cod_risk_bucket(rto_rate):
    if rto_rate <= 0.15:
        return "LOW"
    elif rto_rate <= 0.30:
        return "MEDIUM"
    else:
        return "HIGH"

cod_metrics["cod_risk_bucket"] = cod_metrics["cod_rto_rate"].apply(cod_risk_bucket)

cod_metrics[["sku_id", "region", "cod_rto_rate", "cod_risk_bucket"]].head()


,sku_id,region,cod_rto_rate,cod_risk_bucket
0,SKU0001,east,0.415265,HIGH
1,SKU0001,north,0.385197,HIGH
2,SKU0001,south,0.348164,HIGH
3,SKU0001,west,0.319354,HIGH
4,SKU0002,east,0.412669,HIGH


In [23]:
# mapping regions to warehouse 
cod_metrics["warehouse_id"] = cod_metrics["region"]

cod_metrics.head()


,sku_id,region,total_orders,cod_orders,cod_rto_orders,cod_share,cod_rto_rate,cod_success_rate,cod_risk_bucket,warehouse_id
0,SKU0001,east,13921,6957,2889,0.499749,0.415265,0.584735,HIGH,east
1,SKU0001,north,13296,6485,2498,0.487741,0.385197,0.614803,HIGH,north
2,SKU0001,south,12810,6046,2105,0.471975,0.348164,0.651836,HIGH,south
3,SKU0001,west,10932,5079,1622,0.464599,0.319354,0.680646,HIGH,west
4,SKU0002,east,13136,6567,2710,0.499924,0.412669,0.587331,HIGH,east


In [24]:
# cod policy actions 
def cod_policy_action(bucket):
    if bucket == "LOW":
        return "ALLOW_COD"
    elif bucket == "MEDIUM":
        return "LIMIT_COD"
    else:
        return "DISABLE_COD"

cod_metrics["cod_policy_action"] = cod_metrics["cod_risk_bucket"].apply(cod_policy_action)

# Optional financial risk flag
cod_metrics["financial_risk_flag"] = np.where(
    (cod_metrics["cod_risk_bucket"] == "HIGH") & (cod_metrics["cod_share"] > 0.30),
    True,
    False
)

cod_metrics.head()


,sku_id,region,total_orders,cod_orders,cod_rto_orders,cod_share,cod_rto_rate,cod_success_rate,cod_risk_bucket,warehouse_id,cod_policy_action,financial_risk_flag
0,SKU0001,east,13921,6957,2889,0.499749,0.415265,0.584735,HIGH,east,DISABLE_COD,True
1,SKU0001,north,13296,6485,2498,0.487741,0.385197,0.614803,HIGH,north,DISABLE_COD,True
2,SKU0001,south,12810,6046,2105,0.471975,0.348164,0.651836,HIGH,south,DISABLE_COD,True
3,SKU0001,west,10932,5079,1622,0.464599,0.319354,0.680646,HIGH,west,DISABLE_COD,True
4,SKU0002,east,13136,6567,2710,0.499924,0.412669,0.587331,HIGH,east,DISABLE_COD,True


In [25]:
# cod intelligence output 
cod_intelligence = cod_metrics[
    [
        "sku_id",
        "warehouse_id",
        "cod_share",
        "cod_rto_rate",
        "cod_success_rate",
        "cod_risk_bucket",
        "cod_policy_action",
        "financial_risk_flag"
    ]
].copy()

cod_intelligence.head()


,sku_id,warehouse_id,cod_share,cod_rto_rate,cod_success_rate,cod_risk_bucket,cod_policy_action,financial_risk_flag
0,SKU0001,east,0.499749,0.415265,0.584735,HIGH,DISABLE_COD,True
1,SKU0001,north,0.487741,0.385197,0.614803,HIGH,DISABLE_COD,True
2,SKU0001,south,0.471975,0.348164,0.651836,HIGH,DISABLE_COD,True
3,SKU0001,west,0.464599,0.319354,0.680646,HIGH,DISABLE_COD,True
4,SKU0002,east,0.499924,0.412669,0.587331,HIGH,DISABLE_COD,True


In [ ]:
final_cod_df = cod_intelligence.copy()

# Reorder columns for API / CSV friendliness
final_cod_df = final_cod_df[
    [
        "sku_id",
        "warehouse_id",
        "cod_share",
        "cod_rto_rate",
        "cod_success_rate",
        "cod_risk_bucket",
        "cod_policy_action",
        "financial_risk_flag"
    ]
]

final_cod_df.head()


,sku_id,warehouse_id,cod_share,cod_rto_rate,cod_success_rate,cod_risk_bucket,cod_policy_action,financial_risk_flag
0,SKU0001,east,0.499749,0.415265,0.584735,HIGH,DISABLE_COD,True
1,SKU0001,north,0.487741,0.385197,0.614803,HIGH,DISABLE_COD,True
2,SKU0001,south,0.471975,0.348164,0.651836,HIGH,DISABLE_COD,True
3,SKU0001,west,0.464599,0.319354,0.680646,HIGH,DISABLE_COD,True
4,SKU0002,east,0.499924,0.412669,0.587331,HIGH,DISABLE_COD,True


In [29]:
import os

os.makedirs("../artifacts", exist_ok=True)

cod_path = "../artifacts/cod_intelligence.csv"
final_cod_df.to_csv(cod_path, index=False)

print(f"COD Intelligence saved to {cod_path}")


COD Intelligence saved to ../artifacts/cod_intelligence.csv
